### I just want to know how context handler working

In [27]:
import os
import sys
import django

# Add project root to path
sys.path.insert(0, r'D:\Rushal\VS_code\Python\ProjectIII\backend')

# Set settings module
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'backend.settings')
django.setup()

# Now imports works
from accounts.models import UserProfile
from chats.models import ChatSession
from django.db import connection
import ollama
from ai_app.ai.utils.message import system_talking, user_talking
from ai_app.ai.utils.embeddings import embed_text

In [28]:
def ai(msg,model="llama3.2:latest",tools='auto'):
    response = ollama.chat(
            model=model,
            messages=msg,
            options={
                "temperature": 0.1,      # Creative but not wild
                # "top_p": 0.9,            # Standard
                # "num_predict": 200,      # Short story (~200 tokens)
                # "num_ctx": 4096,         # Enough for context
                # "repeat_penalty": 1.1,   # Avoid repetition
                # "seed": 123,             # Reproducible
                # "mirostat": 1            # Adaptive creativity
            }
            )
    return response["message"]["content"]

In [8]:

def _default_question(role, difficulty):
	print(
		f"[INTERVIEW-RAG] FALLBACK generation started role={role!r} "
		f"difficulty={difficulty!r}",
		flush=True,
	)
	prompt = (
		"Generate one concise interview question only. Do not add an answer, "
		f"markdown, numbering, or explanation. The target role is {role} and "
		f"the difficulty is {difficulty}."
	)
	try:
		question = str(ai([system_talking(prompt)])).strip()
		question = question.splitlines()[0].strip(" -*0123456789.:")
		if question:
			print(
				f"[INTERVIEW-RAG] FALLBACK generated question={question!r}",
				flush=True,
			)
			return question
	except Exception as error:
		print(f"[context_handler] Default question generation failed: {error}", flush=True)
	fallback = f"Describe a recent {difficulty}-level problem you solved as a {role}."
	print(f"[INTERVIEW-RAG] FALLBACK static question={fallback!r}", flush=True)
	return fallback

In [9]:
def _role_for_session(user, session):
	if session.interview_role:
		return session.interview_role

	profile = UserProfile.objects.filter(user=user).only("target_role").first()
	role = profile.target_role.strip() if profile and profile.target_role else ""
	if role:
		session.interview_role = role
		session.save(update_fields=["interview_role", "updated_at"])
	return role


In [10]:

def _retrieve_questions(session, role, query_embedding, top_k, candidate_ids=None):
	candidate_filter = ""
	params = [query_embedding, role, session.current_difficulty, session.id]
	if candidate_ids:
		placeholders = ", ".join(["%s"] * len(candidate_ids))
		candidate_filter = f"AND iq.id IN ({placeholders})"
		params.extend(candidate_ids)
	params.extend([query_embedding, top_k])

	with connection.cursor() as cur:
		cur.execute(f"""
			SELECT
				iq.id, iq.question, iq.answer, iq.role, iq.experience,
				iq.difficulty, iq.keywords, iq.source_id,
				iq.embedding <=> %s::vector AS distance
			FROM common_interviewquestions iq
			WHERE LOWER(iq.role) = LOWER(%s)
			  AND iq.difficulty = %s
			  AND iq.id NOT IN (
				  SELECT asked_questions_id
				  FROM chats_interviewfeedback
				  WHERE session_id = %s AND asked_questions_id IS NOT NULL
			  )
			  {candidate_filter}
			ORDER BY iq.embedding <=> %s::vector
			LIMIT %s;
		""", params)
		return cur.fetchall()


In [ ]:

def prefetch_next_questions(user, session_id, attempt, top_k=3):
	session = ChatSession.objects.get(id=session_id, user=user, archived=False)
	role = _role_for_session(user, session)
	if not role:
		return {"status": "missing_role", "question_ids": []}

	query_embedding = embed_text(
		f"Interview questions for a {session.current_difficulty} {role}"
	)
	questions = _retrieve_questions(session, role, query_embedding, top_k)
	attempt.prefetched_question_ids = [question[0] for question in questions]
	attempt.save(update_fields=["prefetched_question_ids", "updated_at"])
	return {"status": "success", "question_ids": attempt.prefetched_question_ids}


In [ ]:

def get_next_questions(
	user,
	user_content,
	session_id,
	top_k=3,
	candidate_ids=None,
	query_embedding=None,
	session=None,
):
	if session is None:
		session = ChatSession.objects.get(id=session_id, user=user, archived=False)
	role = _role_for_session(user, session)
	if not role:
		return {"status": "missing_role", "message": "Please provide your target role first."}

	if query_embedding is None:
		query_embedding = embed_text(user_content)
	questions = _retrieve_questions(
		session,
		role,
		query_embedding,
		top_k,
		candidate_ids=candidate_ids,
	)
	if not questions and candidate_ids:
		questions = _retrieve_questions(session, role, query_embedding, top_k)

	if not questions:
		return {
			"status": "success",
			"role": role,
			"generated": True,
			"question": {
				"id": None,
				"question": _default_question(role, session.current_difficulty),
				"answer": "",
				"role": role,
				"experience": "any",
				"difficulty": session.current_difficulty,
				"keywords": [],
				"distance": None,
			},
		}

	selected_question = questions[0]
	return {
		"status": "success",
		"role": role,
		"question": {
			"id": selected_question[0],
			"question": selected_question[1],
			"answer": selected_question[2],
			"role": selected_question[3],
			"experience": selected_question[4],
			"difficulty": selected_question[5],
			"keywords": selected_question[6],
			"distance": selected_question[8],
			"source_id": selected_question[7],
		},
	}

In [37]:

role='full stack'
name='rushal'
skills=['react','mern stack','database','cyber-security']


print("INTERVIEW LL BE START BUT PLEASE MAKE SURE U HAVE DEFINED THE REQUIRED THINGS")

prompt = f"""
you are a professional interviewer at XYZ company. Act as a professional HR that is seeking for a perfect person to work at your company
the candidate details are :
name : {name if name else "not given"}
role : {role if role else "not given"}
skilss : {skills if len(skills)>0 else "no skills"}

if anything else is not written please question the user about the things so that u can know about and evaluate each answer
"""

INTERVIEW LL BE START BUT PLEASE MAKE SURE U HAVE DEFINED THE REQUIRED THINGS


In [38]:
print(prompt)
message=[]
message.append(system_talking(prompt))


you are a professional interviewer at XYZ company. Act as a professional HR that is seeking for a perfect person to work at your company
the candidate details are :
name : rushal
role : full stack
skilss : ['react', 'mern stack', 'database', 'cyber-security']

if anything else is not written please question the user about the things so that u can know about and evaluate each answer



In [39]:
message.append(user_talking('hello'))

In [40]:
res = ai(msg=message)

In [41]:
print(res)

Hello Rushal! Thank you for coming in today. I'm Rohan, the HR representative at XYZ company. I'll be conducting your interview today.

Before we begin, I'd like to introduce myself and the company. We're a leading tech firm that specializes in innovative solutions and cutting-edge technologies. We're looking for talented individuals like yourself to join our team.

Can you start by telling me a little about yourself? What inspired you to apply for this role, and what do you know about our company?
